In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

import numpy as np

In [2]:
# Load data
housing = fetch_california_housing()
X = housing.data
y = housing.target
print(X.shape, y.shape)

(20640, 8) (20640,)


In [ ]:
#make it dataframe
import pandas as pd
df = pd.DataFrame(X, columns=housing.feature_names)
df['target'] = y

In [6]:
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [7]:
#split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#scale the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [8]:
#convert the data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)


In [11]:
#now simple regression model using pytorch
class SimpleRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear=nn.Linear(8,1)

    def forward(self,x):
        return self.linear(x)
    
model=SimpleRegressionModel()
print(model)

SimpleRegressionModel(
  (linear): Linear(in_features=8, out_features=1, bias=True)
)


In [12]:
#training setup
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

c:\Users\Furqan Khan\AppData\Local\miniconda3\envs\agent_env2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
epochs = 100
for epoch in range(epochs):
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 0.5182
Epoch [20/100], Loss: 0.4061
Epoch [30/100], Loss: 0.4499
Epoch [40/100], Loss: 0.2846
Epoch [50/100], Loss: 0.4275
Epoch [60/100], Loss: 0.8258
Epoch [70/100], Loss: 0.3121
Epoch [80/100], Loss: 0.6977
Epoch [90/100], Loss: 0.3528
Epoch [100/100], Loss: 0.5461


In [14]:
#evaluate the model on test set
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor)
    test_loss = criterion(y_pred, y_test_tensor)
    print(f'Test Loss: {test_loss.item():.4f}')

    # Calculate additional metrics
    mae = mean_absolute_error(y_test_tensor.numpy(), y_pred.numpy())
    r2 = r2_score(y_test_tensor.numpy(), y_pred.numpy())
    print(f'Mean Absolute Error: {mae:.4f}')
    print(f'R^2 Score: {r2:.4f}')

Test Loss: 0.5445
Mean Absolute Error: 0.5328
R^2 Score: 0.5845


In [15]:
#test single prediction of test set and their actual value
with torch.no_grad():
    sample_index = 0  # Change this index to test different samples
    sample_input = X_test_tensor[sample_index].unsqueeze(0)  # Add batch dimension
    sample_output = model(sample_input)
    print(f'Sample Input: {X_test_tensor[sample_index].numpy()}')
    print(f'Predicted Output: {sample_output.item():.4f}')
    print(f'Actual Output: {y_test_tensor[sample_index].item():.4f}')

Sample Input: [-1.1550847  -0.2863237  -0.5206858  -0.17174603 -0.03030109  0.06740798
  0.1951      0.28534728]
Predicted Output: 0.7174
Actual Output: 0.4770


In [16]:
#save the model
torch.save(model.state_dict(), 'simple_regression_model.pth')
print("Model saved successfully.")

Model saved successfully.
